# Section 10: AI Agents — Foundations, Architectures & Protocols — Hands-On

### GenAI Decoded by Nij — Building Agents from First Principles

---

In **Sections 1–9**, you built everything an LLM needs to be useful: prompting, RAG, function calling, frameworks, fine-tuning, and deployment. But in every case, **you** decided the steps — the model just executed them.

An **agent** flips that. You give it a *goal*, and it figures out the steps: which tools to call, in what order, how to handle failures, and when to stop.

This notebook builds agents from scratch — no frameworks, just Python + the OpenAI API. You'll:

1. **Build the agent loop** — the `for` loop that makes LLMs autonomous
2. **Implement three architectures** — ReAct, Plan-and-Execute, Reflexion
3. **Build an MCP server** — expose tools via the Model Context Protocol
4. **Connect MCP to your agent** — one server, any client
5. **Compare architectures** — same task, different approaches

By the end, you'll understand what every agent framework (LangGraph, CrewAI, OpenAI Agents SDK) is doing under the hood.


## Setup

**Python 3.10+ recommended.** We'll use:

- **OpenAI** (`gpt-4o-mini`) — main model, matches Sections 1–9
- **Groq** (`llama-3.1-8b-instant`) — fast model for Plan-and-Execute executor
- **mcp** — Model Context Protocol SDK for building tool servers
- **python-dotenv** — environment variable loading

Make sure your `.env` has `OPENAI_API_KEY` and `GROQ_API_KEY`.


In [1]:
import sys, subprocess, importlib.util


pkgs = {"openai": "openai", "groq": "groq", "python-dotenv": "dotenv"}
missing = [p for p, m in pkgs.items() if importlib.util.find_spec(m) is None]
if missing:
    print(f"📦 Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
print(f"✅ All packages ready — Python {sys.version.split()[0]}")


✅ All packages ready — Python 3.12.13


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

for key in ["OPENAI_API_KEY", "GROQ_API_KEY"]:
    status = "✅" if os.getenv(key) else "❌ MISSING"
    print(f"{status}  {key}")


✅  OPENAI_API_KEY
✅  GROQ_API_KEY


In [3]:
from openai import OpenAI
from groq import Groq
import json, time

oai = OpenAI()
groq_client = Groq()

OPENAI_MODEL = "gpt-4o-mini"
GROQ_MODEL = "llama-3.1-8b-instant"

def call_llm(prompt, model=OPENAI_MODEL, system=None, tools=None, messages=None):
    """Unified LLM call helper."""
    if messages is None:
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
    client = groq_client if "llama" in model else oai
    kwargs = {"model": model, "messages": messages}
    if tools:
        kwargs["tools"] = tools
    return client.chat.completions.create(**kwargs).choices[0].message

print(f"✅ OpenAI → {OPENAI_MODEL}")
print(f"✅ Groq   → {GROQ_MODEL}")


✅ OpenAI → gpt-4o-mini
✅ Groq   → llama-3.1-8b-instant


---

## 3. The Agent Loop

This is the single most important pattern in this notebook. Every agent — whether built with LangGraph, CrewAI, or raw Python — runs the same core loop:

1. **Send** the current conversation (including tool results) to the LLM
2. **Check** if the LLM wants to call a tool or give a final answer
3. If **tool call** → execute it, append the result, go back to step 1
4. If **final answer** → return it, exit the loop

The LLM decides when to stop. Your code decides the ceiling.


### 3.1 — Define Tools and Schemas


In [4]:
# ── Three tools our agent can use ──

def search_web(query: str) -> str:
    """Simulate a web search with realistic results."""
    knowledge = {
        "bitcoin price": "Bitcoin is currently trading at $68,420 (June 2026), up 4.2% this week.",
        "abu dhabi population": "Abu Dhabi has a population of approximately 1.57 million (2025 est).",
        "tesla revenue": "Tesla reported $25.7B revenue in Q4 2025, with 8.2% operating margin.",
        "python 3.13": "Python 3.13 released Oct 2024 with JIT compiler and free-threading.",
    }
    for key, val in knowledge.items():
        if key in query.lower():
            return val
    return f"Search for '{query}': No specific data found. Try more specific terms."

def calculate(expression: str) -> str:
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return "Error: invalid characters. Use only numbers and +-*/.()"
        return f"{expression} = {eval(expression)}"
    except Exception as e:
        return f"Error: {e}"

def get_current_date() -> str:
    """Return the current date."""
    from datetime import datetime
    return datetime.now().strftime("Today is %A, %B %d, %Y. Time: %H:%M")

# Tool schemas for the LLM
TOOLS = [
    {"type": "function", "function": {
        "name": "search_web",
        "description": "Search the web for current information about events, prices, stats. Returns text summary.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "Search query — be specific"}
        }, "required": ["query"]}
    }},
    {"type": "function", "function": {
        "name": "calculate",
        "description": "Evaluate a math expression. Examples: '100 * 0.15', '2**10', '(500 - 350) / 500 * 100'",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "Math expression to evaluate"}
        }, "required": ["expression"]}
    }},
    {"type": "function", "function": {
        "name": "get_current_date",
        "description": "Get today's date and current time.",
        "parameters": {"type": "object", "properties": {}}
    }},
]

TOOL_FUNCTIONS = {"search_web": search_web, "calculate": calculate, "get_current_date": get_current_date}
print(f"✅ {len(TOOLS)} tools defined: {list(TOOL_FUNCTIONS.keys())}")
print(f"   Test: {search_web('bitcoin price')}")
print(f"   Test: {calculate('2**10 + 3**5')}")


✅ 3 tools defined: ['search_web', 'calculate', 'get_current_date']
   Test: Bitcoin is currently trading at $68,420 (June 2026), up 4.2% this week.
   Test: 2**10 + 3**5 = 1267


### 3.2 — The Agent Loop

Here it is — the core pattern. Every agent framework wraps these ~35 lines.


In [5]:
def agent(user_question, tools=TOOLS, tool_functions=TOOL_FUNCTIONS, max_iterations=10):
    """A complete agent — from scratch, no framework."""
    messages = [
        {"role": "system", "content":
            "You are a helpful research assistant. "
            "Think step by step. Use tools when you need information you don't have. "
            "When you have enough information, respond directly to the user. "
            "For common knowledge questions, answer directly without tools."},
        {"role": "user", "content": user_question}
    ]

    print(f"🧑 USER: {user_question}")
    print("─" * 60)

    for i in range(max_iterations):
        response = oai.chat.completions.create(
            model=OPENAI_MODEL, messages=messages, tools=tools,
        )
        msg = response.choices[0].message
        messages.append(msg)

        # ── Decision point: tool call or final answer? ──
        if msg.tool_calls:
            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)

                # Safety: validate tool name before execution
                if fn_name not in tool_functions:
                    result = f"Error: tool '{fn_name}' not found. Available: {list(tool_functions.keys())}"
                else:
                    result = tool_functions[fn_name](**fn_args)

                print(f"  🔧 [{i+1}] {fn_name}({fn_args}) → {result[:100]}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
        else:
            # No tool call = agent decided it's done
            print("─" * 60)
            print(f"🤖 ANSWER ({i+1} iterations): {msg.content}")
            return msg.content

    print(f"⚠️  Max iterations ({max_iterations}) reached")
    return "Could not complete within iteration limit."

print("✅ Agent function defined")


✅ Agent function defined


### 3.3 — Test the Agent

Watch how the agent makes different decisions for each query.


In [6]:
# Needs search
agent("What's the current price of Bitcoin and how has it moved this week?")


🧑 USER: What's the current price of Bitcoin and how has it moved this week?
────────────────────────────────────────────────────────────
  🔧 [1] search_web({'query': 'current Bitcoin price and weekly movement'}) → Bitcoin is currently trading at $68,420 (June 2026), up 4.2% this week.
────────────────────────────────────────────────────────────
🤖 ANSWER (2 iterations): The current price of Bitcoin is $68,420, and it has increased by 4.2% this week.


'The current price of Bitcoin is $68,420, and it has increased by 4.2% this week.'

In [ ]:
# Needs calculation
agent("What's a 15% tip on $87.50?")


In [ ]:
# Needs multiple tools (search + calculate)
agent("What's Tesla's Q4 2025 revenue, and what's that per month?")


In [ ]:
# No tools needed — should answer directly
agent("What is the capital of France?")


💡 **Key takeaway:** The agent made different decisions for each query — sometimes one tool, sometimes two in sequence, sometimes none. The `for` loop + `if msg.tool_calls` branch is the entire mechanic. Every agent framework wraps this same pattern with extra features (state, persistence, routing) — but this is the core.


---

## 4. ReAct — Explicit Reasoning Traces

The agent above *is* a ReAct agent — we just didn't make the reasoning visible. Let's add explicit Thought/Action/Observation logging so every decision is traceable.


In [7]:
def react_agent(question, tools=TOOLS, tool_functions=TOOL_FUNCTIONS, max_iter=10):
    """ReAct agent with explicit Thought/Action/Observation traces."""
    messages = [
        {"role": "system", "content":
            "You are a research assistant using the ReAct framework. "
            "For EVERY step: 1) Write 'Thought:' explaining your reasoning, "
            "2) Call a tool if needed, 3) After seeing results think again, "
            "4) When you have enough info, give your final answer."},
        {"role": "user", "content": question}
    ]

    print(f"🧑 QUESTION: {question}")
    print("═" * 60)

    for i in range(max_iter):
        response = oai.chat.completions.create(
            model=OPENAI_MODEL, messages=messages, tools=tools)
        msg = response.choices[0].message
        messages.append(msg)

        if msg.content:
            print(f"\n💭 THOUGHT ({i+1}): {msg.content[:300]}")

        if msg.tool_calls:
            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                result = tool_functions.get(fn_name, lambda **k: "Unknown tool")(**fn_args)
                print(f"⚡ ACTION: {fn_name}({json.dumps(fn_args)})")
                print(f"👁 OBSERVE: {result[:200]}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
        else:
            print(f"\n{'═' * 60}")
            print(f"✅ FINAL ANSWER ({i+1} steps):\n{msg.content}")
            return msg.content

    return "Max iterations reached"

print("✅ ReAct agent defined")


✅ ReAct agent defined


In [9]:
react_agent("What's Abu Dhabi's population, and what's the population density if the city covers 972 sq km?")


🧑 QUESTION: What's Abu Dhabi's population, and what's the population density if the city covers 972 sq km?
════════════════════════════════════════════════════════════
⚡ ACTION: search_web({"query": "Abu Dhabi population 2023"})
👁 OBSERVE: Abu Dhabi has a population of approximately 1.57 million (2025 est).
⚡ ACTION: calculate({"expression": "population/972"})
👁 OBSERVE: Error: invalid characters. Use only numbers and +-*/.()
⚡ ACTION: calculate({"expression": "1570000/972"})
👁 OBSERVE: 1570000/972 = 1615.2263374485597

💭 THOUGHT (3): Thought: I found that Abu Dhabi has a population of approximately 1.57 million people. The area of the city is 972 square kilometers. To find the population density, I divided the population by the area in sq km.

Final answer: Abu Dhabi's population is approximately 1.57 million, and the population

════════════════════════════════════════════════════════════
✅ FINAL ANSWER (3 steps):
Thought: I found that Abu Dhabi has a population of approximately 1.57

"Thought: I found that Abu Dhabi has a population of approximately 1.57 million people. The area of the city is 972 square kilometers. To find the population density, I divided the population by the area in sq km.\n\nFinal answer: Abu Dhabi's population is approximately 1.57 million, and the population density is about 1615 people per square kilometer."

💡 **Key takeaway:** Making reasoning explicit gives you a debuggable trace. When something goes wrong, read the `Thought` step to see *why* the agent chose that action. ReAct's biggest advantage isn't the architecture — it's the debugging strategy.


---

## 5. Plan-and-Execute

ReAct is reactive — one step at a time. Plan-and-Execute is deliberate: a **planner** creates the full roadmap first, then an **executor** follows it step by step. Key trick: use a *stronger model* for planning, *cheaper model* for execution.


In [8]:
def plan_and_execute(question, tools=TOOLS, tool_functions=TOOL_FUNCTIONS):
    """Plan-and-Execute: planner creates roadmap, executor follows it."""

    # Phase 1: PLAN
    plan_msg = call_llm(
        prompt=f"Break this task into 2-5 concrete steps. Return ONLY a JSON array.\n\nTask: {question}",
        system="Return only a JSON array like [\"step1\", \"step2\"]. No other text."
    )
    try:
        plan_text = plan_msg.content.strip()
        if plan_text.startswith("```"):
            plan_text = plan_text.split("\n", 1)[1].rsplit("```", 1)[0]
        steps = json.loads(plan_text)
    except:
        print("⚠️ Plan parsing failed, falling back to ReAct")
        return react_agent(question, tools, tool_functions)

    print(f"🧑 QUESTION: {question}")
    print("═" * 60)
    print(f"📋 PLAN ({len(steps)} steps):")
    for i, step in enumerate(steps):
        print(f"   {i+1}. {step}")
    print("─" * 60)

    # Phase 2: EXECUTE each step
    context = []
    for i, step in enumerate(steps):
        print(f"\n▶ STEP {i+1}: {step}")
        ctx_text = "\n".join(context) if context else "No previous context."

        resp = oai.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": "Execute this step. Use tools if needed. Be concise."},
                {"role": "user", "content": f"Previous results:\n{ctx_text}\n\nCurrent step: {step}"}
            ], tools=tools)
        exec_msg = resp.choices[0].message

        step_msgs = [
            {"role": "system", "content": "Complete this step."},
            {"role": "user", "content": f"Previous:\n{ctx_text}\n\nStep: {step}"},
            exec_msg]

        if exec_msg.tool_calls:
            for tc in exec_msg.tool_calls:
                fn = tc.function.name
                args = json.loads(tc.function.arguments)
                result = tool_functions.get(fn, lambda **k: "Unknown tool")(**args)
                print(f"   🔧 {fn}({args}) → {result[:120]}")
                step_msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
            followup = oai.chat.completions.create(model=OPENAI_MODEL, messages=step_msgs)
            step_result = followup.choices[0].message.content
        else:
            step_result = exec_msg.content

        print(f"   ✓ {step_result[:150]}")
        context.append(f"Step {i+1}: {step_result}")

    # Phase 3: SYNTHESIZE
    print(f"\n{'═' * 60}")
    final = call_llm(
        prompt=f"Question: {question}\n\nResults:\n" + "\n".join(context) + "\n\nSynthesize a clear answer.",
        system="Combine step results into one well-written answer.")
    print(f"✅ FINAL ANSWER:\n{final.content}")
    return final.content

print("✅ Plan-and-Execute agent defined")


✅ Plan-and-Execute agent defined


In [12]:
plan_and_execute("What was Tesla's Q4 2025 revenue? Calculate the monthly revenue, and tell me today's date.")


🧑 QUESTION: What was Tesla's Q4 2025 revenue? Calculate the monthly revenue, and tell me today's date.
════════════════════════════════════════════════════════════
📋 PLAN (3 steps):
   1. Find Tesla's Q4 2025 revenue information
   2. Calculate monthly revenue from the Q4 2025 revenue
   3. Retrieve today's date
────────────────────────────────────────────────────────────

▶ STEP 1: Find Tesla's Q4 2025 revenue information
   🔧 search_web({'query': 'Tesla Q4 2025 revenue forecast'}) → Search for 'Tesla Q4 2025 revenue forecast': No specific data found. Try more specific terms.
   ✓ I couldn't find specific revenue information for Tesla's Q4 2025. It's possible that detailed forecasts for that timeframe are not yet available, as s

▶ STEP 2: Calculate monthly revenue from the Q4 2025 revenue
   🔧 calculate({'expression': 'revenue for Q4 2025 / 3'}) → Error: invalid characters. Use only numbers and +-*/.()
   ✓ To calculate the monthly revenue from Tesla's Q4 2025 revenue, you first need

"As of now, specific revenue information for Tesla's Q4 2025 is not available. Such financial projections are typically released closer to the date based on market conditions and company performance. Therefore, to calculate the monthly revenue from Q4 2025, you would need to have the total revenue for that quarter and then divide it by three, since Q4 consists of the months of October, November, and December.\n\nThe formula for calculating monthly revenue is:\n\\[ \\text{Monthly Revenue} = \\frac{\\text{Q4 2025 Revenue}}{3} \\]\n\nAdditionally, today’s date is June 13, 2026. If you have any further questions or require additional assistance, feel free to ask!"

💡 **Key takeaway:** Plan-and-Execute separates *thinking* from *doing*. The plan is created once, each step executes independently. Best for tasks with 4+ predictable steps — reports, research, data analysis.


---

## 6. Reflexion — Self-Critique Loops

Reflexion adds self-evaluation: the agent produces output, checks if it's correct, reflects on failures, and retries. Most powerful for **verifiable output** — code, math, SQL — where you can test the result.


In [9]:
def reflexion_agent(task, evaluator, max_attempts=3):
    """Reflexion: try → evaluate → reflect → retry."""
    reflections = []
    solution = None

    for attempt in range(1, max_attempts + 1):
        prompt = f"Task: {task}"
        if reflections:
            prompt += "\n\nPrevious reflections:"
            for r in reflections:
                prompt += f"\n- Attempt {r['attempt']}: {r['reflection']}"
            prompt += "\nUse these to improve your solution."

        solution = call_llm(
            prompt=prompt,
            system="You are a Python expert. Return ONLY the code. No markdown fences, no explanation."
        ).content

        print(f"\n{'═' * 60}")
        print(f"🔄 ATTEMPT {attempt}")
        print(f"📝 Solution:\n{solution[:300]}")

        result = evaluator(task, solution)
        status = "✅ PASS" if result["passed"] else "❌ FAIL"
        print(f"\n📊 {status} (Score: {result['score']}/100)")
        print(f"   {result['feedback']}")

        if result["passed"]:
            print(f"\n🎉 Solution accepted on attempt {attempt}!")
            return solution

        reflection = call_llm(
            prompt=f"My solution:\n{solution}\n\nFeedback:\n{result['feedback']}\n\nWhat went wrong? Be specific.",
            system="You are a code reviewer. Identify the exact bug and how to fix it."
        ).content

        reflections.append({"attempt": attempt, "reflection": reflection})
        print(f"\n🪞 Reflection: {reflection[:200]}")

    return solution

print("✅ Reflexion agent defined")


✅ Reflexion agent defined


In [14]:
def code_evaluator(task, solution):
    """Evaluate Python code against test cases."""
    code_text = solution.strip()
    if code_text.startswith("```"):
        code_text = code_text.split("\n", 1)[1].rsplit("```", 1)[0]

    tests = [
        (([3,1,4,1,5,9,2,6], 3), [1,1,2]),
        (([10,20,30], 2), [10,20]),
        (([], 5), []),
        (([7], 0), []),
        (([5,3,8,1,9], 5), [1,3,5,8,9]),
    ]

    try:
        ns = {}
        exec(code_text, ns)
        func = next((v for k, v in ns.items() if callable(v) and k != "__builtins__"), None)
        if not func:
            return {"passed": False, "score": 0, "feedback": "No function found in solution"}

        passed, failures = 0, []
        for args, expected in tests:
            try:
                r = func(*args)
                if r == expected:
                    passed += 1
                else:
                    failures.append(f"  {func.__name__}{args} → {r}, expected {expected}")
            except Exception as e:
                failures.append(f"  {func.__name__}{args} → ERROR: {e}")

        score = int(passed / len(tests) * 100)
        fb = f"{passed}/{len(tests)} tests passed."
        if failures:
            fb += " Failures:\n" + "\n".join(failures)
        return {"passed": passed == len(tests), "score": score, "feedback": fb}

    except SyntaxError as e:
        return {"passed": False, "score": 0, "feedback": f"Syntax error: {e}"}
    except Exception as e:
        return {"passed": False, "score": 0, "feedback": f"Error: {e}"}

# Run Reflexion on a coding task
reflexion_agent(
    task="Write a function top_k_smallest(lst, k) returning the k smallest elements sorted. "
         "Edge cases: empty list→[], k=0→[], k>len→full sorted list.",
    evaluator=code_evaluator,
)



════════════════════════════════════════════════════════════
🔄 ATTEMPT 1
📝 Solution:
def top_k_smallest(lst, k):
    if k == 0 or not lst:
        return []
    return sorted(lst)[:k]

📊 ✅ PASS (Score: 100/100)
   5/5 tests passed.

🎉 Solution accepted on attempt 1!


'def top_k_smallest(lst, k):\n    if k == 0 or not lst:\n        return []\n    return sorted(lst)[:k]'

💡 **Key takeaway:** Reflexion works because reflections accumulate — on attempt 3, the agent knows what failed in attempts 1 and 2. The evaluator is key: without automated testing, Reflexion is just expensive regeneration. Use it for code, SQL, math — anything testable.


---

## 7. MCP — Model Context Protocol

MCP standardises how agents connect to tools and data. Write one server — **any** MCP-compatible client can use it (Claude Desktop, ChatGPT, Cursor, your custom agent).

**What you'll build in this section:**
1. **7.1** — Write a full TechStore MCP server (3 tools + 1 resource + 1 prompt)
2. **7.2** — Inspect tool schemas — what the LLM actually sees
3. **7.3** — Connect to the server and call tools directly
4. **7.4** — Wire MCP into a live agent loop (LLM drives tool calls via MCP)
5. **7.5** — Connect to a real market MCP server (Filesystem MCP)
6. **7.6** — MCP Resources and Prompts in practice

The key difference from Section 2 (function calling): with MCP, the tool server is **a separate process**. Your agent discovers and calls tools at runtime — not hardcoded in your script.


In [ ]:
import sys, subprocess, importlib.util

for pkg in ["mcp", "nest_asyncio"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"📦 Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import nest_asyncio
nest_asyncio.apply()  # Allow asyncio.run() inside Jupyter

print("✅ MCP SDK ready")
print("   mcp — Python SDK for building and connecting to MCP servers")
print("   nest_asyncio — allows async code to run inside Jupyter cells")


### 7.1 — Write the TechStore MCP Server

The server runs as a **separate process**. We write the full source to a file, then connect to it from this notebook via stdio transport.

This server exposes:
- **3 Tools**: `check_inventory`, `search_products`, `get_order_status`
- **1 Resource**: `docs://refund-policy` — a readable policy document
- **1 Prompt**: `analyze_order` — a reusable prompt template

This matches the full implementation walkthrough in the web page (Chapter 12).


In [10]:
import pathlib

# Server source broken into lines to avoid triple-quote nesting issues
_SERVER_LINES = ['from mcp.server import Server', 'from mcp.server.stdio import stdio_server', 'from mcp.types import TextContent, Resource, Prompt, PromptMessage, TextResourceContents', 'import json, asyncio', '', 'server = Server("techstore")', '', '# ── Mock data (replace with real DB calls in production) ──', 'PRODUCTS = {', '    "laptop-pro":     {"name": "TechStore Laptop Pro",      "price": 1299.99, "qty": 23,  "category": "laptops"},', '    "wireless-mouse": {"name": "ErgoClick Wireless Mouse",  "price": 49.99,  "qty": 156, "category": "accessories"},', '    "usb-hub":        {"name": "USB-C Hub 7-in-1",          "price": 34.99,  "qty": 0,   "category": "accessories"},', '    "monitor-4k":     {"name": "UltraView 4K Monitor 27in", "price": 449.99, "qty": 12,  "category": "monitors"},', '    "keyboard-mech":  {"name": "MechType Pro Keyboard",     "price": 129.99, "qty": 67,  "category": "accessories"},', '}', 'ORDERS = {', '    "ORD-10001": {"status": "delivered",  "items": ["laptop-pro"],                 "total": 1299.99, "date": "2025-12-15"},', '    "ORD-10002": {"status": "shipped",    "items": ["wireless-mouse", "usb-hub"],  "total": 84.98,  "date": "2026-01-03"},', '    "ORD-10003": {"status": "processing", "items": ["monitor-4k"],                 "total": 449.99, "date": "2026-01-10"},', '}', '', '# ── Tool 1: Check inventory ──', '@server.tool()', 'async def check_inventory(product_name: str) -> list[TextContent]:', '    """Check if a product is in stock and return its price.', '', '    Use when a customer asks about product availability, stock levels,', "    or pricing for a specific item. Accepts partial names (e.g. 'laptop').", '    """', '    matches = [p for p in PRODUCTS.values() if product_name.lower() in p["name"].lower()]', '    if not matches:', '        return [TextContent(type="text",', '            text=f"No products matching \'{product_name}\'. Available: laptops, accessories, monitors.")]', '    lines = [', '        f"  {p[\'name\']}: ${p[\'price\']:.2f} — {\'In Stock\' if p[\'qty\'] > 0 else \'OUT OF STOCK\'} ({p[\'qty\']} units)"', '        for p in matches', '    ]', '    return [TextContent(type="text", text="\\n".join(lines))]', '', '# ── Tool 2: Search products ──', '@server.tool()', 'async def search_products(query: str = "", category: str = "", in_stock_only: bool = False) -> list[TextContent]:', '    """Search the product catalog by name, category, or availability.', '', '    Use when a customer wants to browse products or find options in a category.', '    Category options: laptops, accessories, monitors.', '    """', '    results = list(PRODUCTS.values())', '    if query:', '        results = [p for p in results if query.lower() in p["name"].lower()]', '    if category:', '        results = [p for p in results if p["category"] == category.lower()]', '    if in_stock_only:', '        results = [p for p in results if p["qty"] > 0]', '    if not results:', '        return [TextContent(type="text", text="No products found matching your search.")]', '    lines = [f"Found {len(results)} product(s):"] + [', '        f"  {p[\'name\']} — ${p[\'price\']:.2f} ({\'In Stock\' if p[\'qty\'] > 0 else \'Out of Stock\'})"', '        for p in results', '    ]', '    return [TextContent(type="text", text="\\n".join(lines))]', '', '# ── Tool 3: Order status ──', '@server.tool()', 'async def get_order_status(order_id: str) -> list[TextContent]:', '    """Look up the current status of a customer order.', '', '    Use when a customer asks about their order, delivery, or shipment.', '    Order IDs have the format ORD-XXXXX.', '    """', '    order_id = order_id.upper().strip()', '    order = ORDERS.get(order_id)', '    if not order:', '        return [TextContent(type="text",', '            text=f"Order {order_id} not found. Please verify the ID (format: ORD-XXXXX).")]', '    items_str = ", ".join(PRODUCTS.get(i, {"name": i})["name"] for i in order["items"])', '    return [TextContent(type="text",', '        text=f"Order {order_id}:\\n"', '             f"  Status: {order[\'status\'].upper()}\\n"', '             f"  Items: {items_str}\\n"', '             f"  Total: ${order[\'total\']:.2f}\\n"', '             f"  Date:  {order[\'date\']}")]', '', '# ── Resource: Refund policy document ──', '@server.list_resources()', 'async def list_resources():', '    return [Resource(uri="docs://refund-policy", name="Refund Policy", mimeType="text/markdown")]', '', '@server.read_resource()', 'async def read_resource(uri: str):', '    if uri == "docs://refund-policy":', '        content = (', '            "# TechStore Refund Policy\\n\\n"', '            "Returns accepted within 30 days of delivery.\\n"', '            "Items must be in original packaging with all accessories.\\n"', '            "Refund processed within 5-7 business days to original payment method.\\n"', '            "Digital products: no refunds after download or activation.\\n"', '            "Shipping costs are non-refundable unless the item was defective.\\n"', '        )', '        return [TextResourceContents(uri=uri, mimeType="text/markdown", text=content)]', '    raise ValueError(f"Unknown resource: {uri}")', '', '# ── Prompt: Reusable order analysis template ──', '@server.list_prompts()', 'async def list_prompts():', '    return [Prompt(', '        name="analyze_order",', '        description="Analyse an order for potential issues or anomalies",', '        arguments=[{"name": "order_id", "description": "Order ID to analyse", "required": True}],', '    )]', '', '@server.get_prompt()', 'async def get_prompt(name: str, arguments: dict):', '    if name == "analyze_order":', '        order_id = arguments.get("order_id", "")', '        order = ORDERS.get(order_id.upper(), {"status": "unknown", "items": [], "total": 0})', '        return [PromptMessage(', '            role="user",', '            content=TextContent(type="text",', '                text=f"Analyse order {order_id}:\\n{json.dumps(order, indent=2)}\\n\\n"', '                     f"Check for: unusual quantities, price anomalies, delivery risk.\\n"', '                     f"Rate each risk LOW / MEDIUM / HIGH."),', '        )]', '    raise ValueError(f"Unknown prompt: {name}")', '', '# ── Entry point ──', 'if __name__ == "__main__":', '    async def main():', '        async with stdio_server() as (r, w):', '            await server.run(r, w)', '    asyncio.run(main())', '']

pathlib.Path("techstore_mcp_server.py").write_text("\n".join(_SERVER_LINES))

print("✅ techstore_mcp_server.py written")
print("   Tools:     check_inventory, search_products, get_order_status")
print("   Resources: docs://refund-policy")
print("   Prompts:   analyze_order")
print()
print("The server runs as a separate process via stdio transport.")
print("In production: swap PRODUCTS/ORDERS dicts for real DB queries.")


✅ techstore_mcp_server.py written
   Tools:     check_inventory, search_products, get_order_status
   Resources: docs://refund-policy
   Prompts:   analyze_order

The server runs as a separate process via stdio transport.
In production: swap PRODUCTS/ORDERS dicts for real DB queries.


### 7.2 — Inspect Tool Schemas

Before running the agent, let's see what the MCP protocol actually exposes — the schemas the LLM reads to decide which tools to use.


In [3]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import sys

async def inspect_server():
    params = StdioServerParameters(command=sys.executable, args=["techstore_mcp_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ── Tools ──
            tools = await session.list_tools()
            print(f"🔌 Connected — {len(tools.tools)} tools discovered:\n")
            for t in tools.tools:
                print(f"  📦 {t.name}")
                print(f"     Description: {(t.description or '').split(chr(10))[0]}")
                props = t.inputSchema.get("properties", {}) if t.inputSchema else {}
                print(f"     Parameters: {list(props.keys()) or '(none)'}")
                print()

            # ── Resources ──
            resources = await session.list_resources()
            print(f"📄 {len(resources.resources)} resource(s):")
            for r in resources.resources:
                print(f"   {r.uri} — {r.name} ({r.mimeType})")

            # ── Prompts ──
            prompts = await session.list_prompts()
            print(f"\n💬 {len(prompts.prompts)} prompt template(s):")
            for p in prompts.prompts:
                print(f"   {p.name} — {p.description}")
            
            # ── Show full schema for one tool ──
            print("\n─── Full schema for check_inventory ───")
            import json
            inv_tool = next(t for t in tools.tools if t.name == "check_inventory")
            print(json.dumps({
                "name": inv_tool.name,
                "description": inv_tool.description,
                "inputSchema": inv_tool.inputSchema,
            }, indent=2))


await inspect_server()


🔌 Connected — 3 tools discovered:

  📦 check_inventory
     Description: Check if a product is in stock and return its price.
     Parameters: ['product_name']

  📦 search_products
     Description: Search the product catalog by name, category, or availability.
     Parameters: ['query', 'category', 'in_stock_only']

  📦 get_order_status
     Description: Look up the current status of a customer order.
     Parameters: ['order_id']

📄 1 resource(s):
   docs://refund-policy — Refund Policy (text/markdown)

💬 1 prompt template(s):
   analyze_order — Analyse an order for potential issues or anomalies.

─── Full schema for check_inventory ───
{
  "name": "check_inventory",
  "description": "Check if a product is in stock and return its price.\n\n    Use when a customer asks about product availability, stock levels,\n    or pricing for a specific item. Accepts partial names (e.g. 'laptop').\n    ",
  "inputSchema": {
    "properties": {
      "product_name": {
        "title": "Product Name

### 7.3 — Call Tools and Read Resources Directly

With the protocol confirmed, let's call each tool and read the resource — exactly as a client agent would.


In [5]:
async def test_mcp_primitives():
    import json
    params = StdioServerParameters(command=sys.executable, args=["techstore_mcp_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            print("═" * 55)
            print("  TOOL CALLS")
            print("═" * 55)

            # Tool 1: check_inventory
            r = await session.call_tool("check_inventory", {"product_name": "laptop"})
            print(f"\n🔧 check_inventory('laptop'):\n{r.content[0].text}")

            # Tool 2: search_products (with filters)
            r = await session.call_tool("search_products", {"category": "accessories", "in_stock_only": True})
            print(f"\n🔧 search_products(category='accessories', in_stock_only=True):\n{r.content[0].text}")

            # Tool 3: get_order_status
            r = await session.call_tool("get_order_status", {"order_id": "ORD-10002"})
            print(f"\n🔧 get_order_status('ORD-10002'):\n{r.content[0].text}")

            # Tool — graceful error handling
            r = await session.call_tool("get_order_status", {"order_id": "ORD-99999"})
            print(f"\n🔧 get_order_status('ORD-99999') [not found]:\n{r.content[0].text}")

            print("\n" + "═" * 55)
            print("  RESOURCE READ")
            print("═" * 55)
            resource = await session.read_resource("docs://refund-policy")
            print(f"\n📄 docs://refund-policy:\n{resource.contents[0].text}")

await test_mcp_primitives()


═══════════════════════════════════════════════════════
  TOOL CALLS
═══════════════════════════════════════════════════════

🔧 check_inventory('laptop'):
  TechStore Laptop Pro: $1299.99 — In Stock (23 units)

🔧 search_products(category='accessories', in_stock_only=True):
Found 2 product(s):
  ErgoClick Wireless Mouse — $49.99 (In Stock)
  MechType Pro Keyboard — $129.99 (In Stock)

🔧 get_order_status('ORD-10002'):
Order ORD-10002:
  Status: SHIPPED
  Items: ErgoClick Wireless Mouse, USB-C Hub 7-in-1
  Total: $84.98
  Date:  2026-01-03

🔧 get_order_status('ORD-99999') [not found]:
Order ORD-99999 not found. Please verify the ID (format: ORD-XXXXX).

═══════════════════════════════════════════════════════
  RESOURCE READ
═══════════════════════════════════════════════════════

📄 docs://refund-policy:
# TechStore Refund Policy

Returns accepted within 30 days of delivery.
Items must be in original packaging with all accessories.
Refund processed within 5-7 business days to original paym

💡 **Key takeaway:** The MCP client/server interaction is pure JSON-RPC over stdio. `list_tools()` gives you the schemas. `call_tool(name, args)` runs the tool. `read_resource(uri)` fetches data. Your agent code doesn't need to know *how* the tool works — just what inputs to pass.

Notice the error handling: `get_order_status` on a missing ID returns a **helpful message**, not a Python exception. That message lands in the LLM's context and lets it ask the user to correct the ID. This is the MCP error contract.


### 7.4 — Wire MCP into a Live Agent Loop

This is the key integration: the LLM decides *which tool to call* based on the schemas it discovers. We translate MCP tool schemas into OpenAI function-calling format, run the ReAct loop, and dispatch calls back through the MCP session.

The agent never has hardcoded tool implementations — it discovers and calls everything through the MCP protocol.


In [12]:
async def run_mcp_agent(question: str):
    """
    Full agent loop driven by MCP:
    1. Connect to MCP server
    2. Discover tools → convert to OpenAI schemas
    3. Run ReAct loop — LLM calls tools via MCP
    """
    import json

    params = StdioServerParameters(command=sys.executable, args=["techstore_mcp_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Step 1: Discover tools from MCP server
            mcp_tools_list = await session.list_tools()
            
            # Step 2: Convert MCP schemas → OpenAI function-calling format
            openai_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description or "",
                        "parameters": t.inputSchema or {"type": "object", "properties": {}},
                    }
                }
                for t in mcp_tools_list.tools
            ]
            print(f"🔌 MCP server connected — {len(openai_tools)} tools ready")
            print(f"   {[t['function']['name'] for t in openai_tools]}\n")

            # Step 3: Agent loop (same pattern as Section 3)
            messages = [
                {"role": "system", "content":
                    "You are a TechStore assistant. Help customers with product info, "
                    "order status, and store policies. Use the available tools to look "
                    "up real data. For policy questions, check the refund policy resource."},
                {"role": "user", "content": question}
            ]

            print(f"🧑 QUESTION: {question}")
            print("─" * 55)

            for iteration in range(8):
                response = oai.chat.completions.create(
                    model=OPENAI_MODEL,
                    messages=messages,
                    tools=openai_tools,
                )
                msg = response.choices[0].message
                messages.append(msg)

                if not msg.tool_calls:
                    # LLM decided it has enough — final answer
                    print(f"\n✅ ANSWER: {msg.content}")
                    return msg.content

                # Execute each tool call via MCP
                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments)
                    print(f"   ⚙️  [{iteration+1}] {tc.function.name}({args})")
                    
                    result = await session.call_tool(tc.function.name, args)
                    result_text = result.content[0].text if result.content else "No result"
                    print(f"       → {result_text[:120]}")

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": result_text,
                    })

            print("⚠️  Max iterations reached")

# Test 1: Single-tool question
await run_mcp_agent("Is the TechStore Laptop Pro in stock?")


🔌 MCP server connected — 3 tools ready
   ['check_inventory', 'search_products', 'get_order_status']

🧑 QUESTION: Is the TechStore Laptop Pro in stock?
───────────────────────────────────────────────────────
   ⚙️  [1] check_inventory({'product_name': 'TechStore Laptop Pro'})
       →   TechStore Laptop Pro: $1299.99 — In Stock (23 units)

✅ ANSWER: The TechStore Laptop Pro is currently in stock, priced at $1299.99, with 23 units available.


'The TechStore Laptop Pro is currently in stock, priced at $1299.99, with 23 units available.'

In [ ]:
# Test 2: Multi-tool — order + policy
asyncio.run(run_mcp_agent(
    "My order ORD-10001 was delivered but I want to return it. What's the return policy and can I still return it?"
))


In [ ]:
# Test 3: Category browsing
asyncio.run(run_mcp_agent("Show me all in-stock accessories under $100"))


💡 **Key takeaway:** The agent doesn't know *how* `check_inventory` works — it just calls it by name with the arguments the LLM generated. The MCP server handles execution. Swap the server for a production database, and **zero agent code changes**.

This is the MCP promise: write your tool logic once in the server, and any LLM client can use it — Claude Desktop, ChatGPT, your custom loop, all without modification.


### 7.5 — Using a Real Market MCP Server

You don't have to write every server yourself. There's a growing ecosystem of pre-built MCP servers you can plug into your agents immediately.

**The `mcp-server-filesystem`** is one of the most useful: it gives any MCP agent read/write access to files on your local machine. It's maintained by Anthropic and available on npm.

> ℹ️ **Prerequisite:** You need Node.js installed (`node --version`).  
> Install the server once: `npm install -g @modelcontextprotocol/server-filesystem`

We'll connect to it and give our agent the ability to **read and write files** — without writing a single line of server code.


In [ ]:
import shutil, subprocess

# Check prerequisites
node_path = shutil.which("node")
npx_path = shutil.which("npx")

print(f"Node.js: {'✅ ' + subprocess.check_output(['node', '--version']).decode().strip() if node_path else '❌ Not found — install from nodejs.org'}")
print(f"npx:     {'✅ found' if npx_path else '❌ Not found'}")

if node_path and npx_path:
    print("\n✅ Ready to connect to @modelcontextprotocol/server-filesystem")
    print("   This MCP server gives agents read/write access to a directory")
    print("   Same MCP protocol — zero changes to how the agent calls tools")
else:
    print("\n⚠️  Install Node.js to run this section: https://nodejs.org")
    print("   The pattern is identical to the custom server above.")


In [ ]:
import os, asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Expose the current working directory to the agent
SANDBOX_DIR = os.path.abspath("mcp_sandbox")
os.makedirs(SANDBOX_DIR, exist_ok=True)

# Write a couple of test files
with open(os.path.join(SANDBOX_DIR, "sales_report.txt"), "w") as f:
    f.write("Q1 2026 Sales Report\n====================\n")
    f.write("Laptops:      $320,000\n")
    f.write("Monitors:     $89,500\n")
    f.write("Accessories:  $47,200\n")
    f.write("Total:        $456,700\n")

with open(os.path.join(SANDBOX_DIR, "todo.txt"), "w") as f:
    f.write("1. Review inventory levels\n2. Update product pricing\n3. Contact supplier for USB hubs\n")

print(f"📁 Sandbox created: {SANDBOX_DIR}")
print(f"   Files: {os.listdir(SANDBOX_DIR)}")

async def explore_filesystem_mcp():
    """Connect to the market filesystem MCP server and discover its tools."""
    import shutil
    if not shutil.which("npx"):
        print("⚠️  Node.js/npx not found. Skipping live demo.")
        print("\n   When installed, connect like this:")
        print(f"   StdioServerParameters(command='npx', args=['-y', '@modelcontextprotocol/server-filesystem', '{SANDBOX_DIR}'])")
        return

    params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", SANDBOX_DIR],
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print(f"\n🔌 Filesystem MCP server — {len(tools.tools)} tools:\n")
            for t in tools.tools:
                print(f"  📦 {t.name}: {(t.description or '').split(chr(10))[0][:70]}")

            # List files in sandbox
            print("\n─── list_directory ───")
            r = await session.call_tool("list_directory", {"path": SANDBOX_DIR})
            print(r.content[0].text)

            # Read a file
            print("\n─── read_file ───")
            r = await session.call_tool("read_file", {"path": os.path.join(SANDBOX_DIR, "sales_report.txt")})
            print(r.content[0].text)

asyncio.run(explore_filesystem_mcp())


In [ ]:
async def run_filesystem_agent(question: str):
    """Agent with filesystem access via market MCP server."""
    import json, shutil

    if not shutil.which("npx"):
        print("⚠️  npx not found — showing expected output only.")
        print(f"\n   Question: {question}")
        print("   Expected: Agent reads sales_report.txt, calculates percentages, writes summary.")
        return

    params = StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", SANDBOX_DIR],
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Discover tools from the market server
            mcp_tools_list = await session.list_tools()
            openai_tools = [
                {"type": "function", "function": {
                    "name": t.name,
                    "description": t.description or "",
                    "parameters": t.inputSchema or {"type": "object", "properties": {}},
                }}
                for t in mcp_tools_list.tools
            ]

            messages = [
                {"role": "system", "content":
                    f"You are a file analysis assistant. The working directory is {SANDBOX_DIR}. "
                    "Use the filesystem tools to read, analyse, and write files as requested."},
                {"role": "user", "content": question}
            ]

            print(f"🧑 QUESTION: {question}")
            print(f"   (Using @modelcontextprotocol/server-filesystem — 0 custom server lines)")
            print("─" * 55)

            for iteration in range(8):
                response = oai.chat.completions.create(
                    model=OPENAI_MODEL, messages=messages, tools=openai_tools)
                msg = response.choices[0].message
                messages.append(msg)

                if not msg.tool_calls:
                    print(f"\n✅ ANSWER: {msg.content}")
                    return

                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments)
                    fname = tc.function.name
                    # Truncate path display for readability
                    display_args = {k: (v.replace(SANDBOX_DIR, "SANDBOX") if isinstance(v, str) else v)
                                    for k, v in args.items()}
                    print(f"   ⚙️  [{iteration+1}] {fname}({display_args})")
                    result = await session.call_tool(fname, args)
                    result_text = result.content[0].text if result.content else "No result"
                    print(f"       → {result_text[:100]}")
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result_text})

asyncio.run(run_filesystem_agent(
    "Read the sales report, calculate each category's percentage of total revenue, "
    "and write a one-paragraph summary to summary.txt"
))


💡 **Key takeaway:** You just wired a **market MCP server** into your agent in ~30 lines of agent code. The server (`@modelcontextprotocol/server-filesystem`) was built by Anthropic. You just consumed it — no implementation required.

This is the MCP ecosystem in action. Other popular market servers:
- `@modelcontextprotocol/server-github` — read repos, issues, PRs
- `@modelcontextprotocol/server-postgres` — query PostgreSQL
- `@modelcontextprotocol/server-brave-search` — live web search
- `@modelcontextprotocol/server-slack` — read/send Slack messages

All use the exact same client code. The only change is the `StdioServerParameters`.


### 7.6 — MCP Resources and Prompts in Practice

Beyond tools, MCP servers expose two more primitives. Here's when and how to use them.


In [ ]:
async def demo_resources_and_prompts():
    """
    Show MCP Resources (read policy doc) and Prompts (get template)
    from our TechStore server.
    """
    import json
    params = StdioServerParameters(command=sys.executable, args=["techstore_mcp_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            print("═" * 55)
            print("  MCP RESOURCE — reading a policy document")
            print("═" * 55)
            # Resources are fetched by the host/client (not triggered by the LLM)
            # Typically pre-loaded as context before the agent runs
            resource = await session.read_resource("docs://refund-policy")
            print(resource.contents[0].text)

            print()
            print("═" * 55)
            print("  MCP PROMPT — fetching a reusable prompt template")
            print("═" * 55)
            # Prompts are curated templates the server exposes
            # The host substitutes them in as the conversation start
            prompt_result = await session.get_prompt("analyze_order", {"order_id": "ORD-10002"})
            print(f"Template returned ({len(prompt_result.messages)} message(s)):")
            print(prompt_result.messages[0].content.text)

            print()
            print("═" * 55)
            print("  WHEN TO USE EACH PRIMITIVE")
            print("═" * 55)
            comparison = [
                ("Tools",     "Agent-triggered",  "check_inventory, get_order_status",  "Action, real-time data"),
                ("Resources", "Host/user-loaded",  "docs://refund-policy",               "Background context, read-only docs"),
                ("Prompts",   "Host-injected",     "analyze_order",                      "Standardised task templates"),
            ]
            header = f"{'Primitive':<12} {'Who triggers':<20} {'Example':<35} {'Use when'}"
            print(header)
            print("─" * len(header))
            for row in comparison:
                print(f"{row[0]:<12} {row[1]:<20} {row[2]:<35} {row[3]}")

asyncio.run(demo_resources_and_prompts())


💡 **Key takeaway — three MCP primitives, three use cases:**

| Primitive | The LLM... | Best for |
|-----------|-----------|----------|
| **Tools** | Triggers autonomously | Real-time actions, data lookup |
| **Resources** | Reads as background context | Policies, catalogs, config docs |
| **Prompts** | Receives as conversation template | Standardised multi-step interactions |

Start with tools. Add resources when the agent needs stable background knowledge. Add prompts when you want repeatable structured workflows.


---

## 8. A2A — Agent-to-Agent Protocol

MCP connects an agent **down** to tools. A2A connects agents **across** to each other.

If you have a LangGraph agent great at research and a CrewAI agent great at writing, how do they collaborate without custom glue code? A2A solves that — it's the same M×N standardisation story as MCP, but for agent-to-agent communication.

**Three core concepts:**
- **Agent Cards** — machine-readable capability declarations (like an OpenAPI spec for agents)
- **Tasks** — units of work with a lifecycle (`submitted → working → completed`)
- **Payloads** — structured data exchanged between agents (Parts and Artifacts)

**What you'll build:**
1. **8.1** — A minimal A2A-style agent server (HTTP + JSON-RPC)
2. **8.2** — An orchestrator that discovers capabilities and delegates a task
3. **8.3** — End-to-end: a research task delegated cross-framework style


### 8.1 — Build a Minimal A2A Agent Server

In production, A2A runs over HTTP. For this notebook we simulate the protocol in-process to keep things dependency-light — the message structures and lifecycle are identical.

We'll build a `ResearchAgent` that:
- Publishes an **Agent Card** at `/.well-known/agent.json`
- Accepts **Tasks** with a `submitted → working → completed` lifecycle
- Uses our tools to do real research
- Returns structured **Artifacts**


In [ ]:
import json, asyncio, uuid, time
from dataclasses import dataclass, field
from typing import Optional

# ── A2A data structures (mirrors the official spec) ──

@dataclass
class AgentCard:
    name: str
    description: str
    url: str
    skills: list[dict]
    capabilities: dict = field(default_factory=lambda: {"streaming": False, "pushNotifications": False})
    
    def to_dict(self):
        return {
            "name": self.name, "description": self.description,
            "url": self.url, "skills": self.skills,
            "capabilities": self.capabilities,
            "authentication": {"schemes": ["none"]},
        }

@dataclass
class TaskStatus:
    state: str  # submitted | working | input-required | completed | failed
    message: Optional[str] = None

@dataclass
class Task:
    id: str
    input_text: str
    status: TaskStatus = field(default_factory=lambda: TaskStatus("submitted"))
    artifacts: list[dict] = field(default_factory=list)
    history: list[dict] = field(default_factory=list)


# ── Research Agent (A2A server) ──

class ResearchAgent:
    """A specialised research agent that exposes the A2A protocol."""

    CARD = AgentCard(
        name="TechStore Research Agent",
        description="Researches product information, pricing, and inventory using real-time tools",
        url="https://research-agent.techstore.example.com",
        skills=[
            {"name": "product_research",
             "description": "Research product availability, pricing, and specifications",
             "input_modes": ["text"], "output_modes": ["text"]},
            {"name": "order_investigation",
             "description": "Investigate order status and related issues",
             "input_modes": ["text"], "output_modes": ["text"]},
        ],
    )

    def get_agent_card(self) -> dict:
        """Equivalent to GET /.well-known/agent.json"""
        return self.CARD.to_dict()

    async def send_task(self, task_input: str, task_id: str = None) -> Task:
        """Equivalent to POST /tasks/send — accepts a task and processes it."""
        task = Task(
            id=task_id or f"task-{uuid.uuid4().hex[:8]}",
            input_text=task_input,
            status=TaskStatus("submitted"),
        )
        task.history.append({"role": "user", "text": task_input, "timestamp": time.time()})
        
        # Process the task (in production: this runs async in the background)
        await self._process_task(task)
        return task

    async def _process_task(self, task: Task):
        """Internal: processes the task using tools."""
        task.status = TaskStatus("working", "Analysing request and running tools...")
        
        # Use our existing agent infrastructure to handle the research
        messages = [
            {"role": "system", "content":
                "You are a TechStore research specialist. Research the user's question "
                "thoroughly using available tools. Return a structured, complete answer."},
            {"role": "user", "content": task.input_text}
        ]

        for _ in range(6):
            response = oai.chat.completions.create(
                model=OPENAI_MODEL, messages=messages, tools=TOOLS)
            msg = response.choices[0].message
            messages.append(msg)

            if not msg.tool_calls:
                # Research complete — package as artifact
                task.status = TaskStatus("completed")
                task.artifacts.append({
                    "name": "research_result",
                    "parts": [{"type": "text", "text": msg.content}],
                    "metadata": {"tools_used": len([m for m in messages if m.get("role") == "tool"])},
                })
                task.history.append({
                    "role": "agent", "text": msg.content, "timestamp": time.time()})
                return

            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                fn = TOOL_FUNCTIONS.get(tc.function.name)
                result = fn(**args) if fn else f"Tool {tc.function.name} not found"
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

        task.status = TaskStatus("failed", "Exceeded maximum processing steps")


research_agent = ResearchAgent()

# Show the Agent Card
print("📋 AGENT CARD (served at /.well-known/agent.json):")
print(json.dumps(research_agent.get_agent_card(), indent=2))


### 8.2 — Orchestrator: Discover and Delegate

The orchestrator reads the Agent Card to discover what the research agent can do, then delegates a task and waits for completion — without knowing anything about the agent's implementation.


In [ ]:
class OrchestratorAgent:
    """Orchestrates work by delegating to specialised agents via A2A."""

    def __init__(self, agents: list):
        self.agents = agents  # In production: fetched via HTTP from agent URLs

    def discover_agents(self) -> list[dict]:
        """Read Agent Cards from all known agents."""
        return [agent.get_agent_card() for agent in self.agents]

    def find_agent_for_skill(self, skill_name: str):
        """Find the right agent for a required skill."""
        for agent in self.agents:
            card = agent.get_agent_card()
            if any(s["name"] == skill_name for s in card.get("skills", [])):
                return agent
        return None

    async def delegate_task(self, task_input: str, required_skill: str) -> Task:
        """
        Core A2A flow:
        1. Find an agent with the right skill (via Agent Card)
        2. Send the task
        3. Wait for completion
        4. Return the artifact
        """
        agent = self.find_agent_for_skill(required_skill)
        if not agent:
            raise ValueError(f"No agent found with skill: {required_skill}")

        card = agent.get_agent_card()
        print(f"   🔍 Found agent: {card['name']}")
        print(f"   📤 Delegating task via A2A...")

        task = await agent.send_task(task_input)
        return task

    async def handle_complex_request(self, request: str):
        """
        Multi-step orchestration: plan → delegate sub-tasks → synthesise.
        Shows A2A used for real cross-agent collaboration.
        """
        print(f"\n🧑 COMPLEX REQUEST: {request}")
        print("═" * 60)

        # Step 1: Planner decides what skills are needed
        plan_response = call_llm(
            prompt=f"What research skills are needed to answer: '{request}'?\n"
                   f"Available skills: product_research, order_investigation\n"
                   f"Return a JSON object: {{\"skills\": [\"skill1\", ...], \"sub_tasks\": [\"task1\", ...]}}",
            system="Return only valid JSON, no markdown."
        )
        try:
            plan_text = plan_response.content.strip().lstrip("```json").rstrip("```").strip()
            plan = json.loads(plan_text)
        except:
            plan = {"skills": ["product_research"], "sub_tasks": [request]}

        print(f"\n📋 PLAN: {plan.get('sub_tasks', [request])}")
        print(f"   Skills required: {plan.get('skills', ['product_research'])}")
        
        # Step 2: Delegate sub-tasks
        results = []
        for i, (task_text, skill) in enumerate(zip(
            plan.get("sub_tasks", [request]),
            plan.get("skills", ["product_research"] * len(plan.get("sub_tasks", [request])))
        )):
            print(f"\n[Sub-task {i+1}] {task_text[:60]}...")
            task = await self.delegate_task(task_text, skill)
            print(f"   Status: {task.status.state.upper()}")
            if task.artifacts:
                artifact_text = task.artifacts[0]["parts"][0]["text"]
                print(f"   Artifact: {artifact_text[:100]}...")
                results.append(artifact_text)

        # Step 3: Synthesise
        if len(results) > 1:
            print(f"\n🔗 SYNTHESISING {len(results)} sub-task results...")
            synthesis = call_llm(
                prompt=f"Original request: {request}\n\nResearch results:\n" +
                       "\n---\n".join(results) +
                       "\n\nSynthesise into a single coherent answer.",
            )
            print(f"\n✅ FINAL ANSWER: {synthesis.content}")
        else:
            print(f"\n✅ FINAL ANSWER: {results[0] if results else 'No results'}")


# ── Run it ──
orchestrator = OrchestratorAgent(agents=[research_agent])

print("=== Agent Discovery ===")
cards = orchestrator.discover_agents()
print(f"Discovered {len(cards)} agent(s):")
for card in cards:
    print(f"  • {card['name']}: {len(card['skills'])} skill(s) — {[s['name'] for s in card['skills']]}")


### 8.3 — End-to-End: A2A Task Delegation


In [ ]:
# Test 1: Simple delegation — single skill
print("TEST 1: SINGLE SKILL DELEGATION")
print("─" * 60)
task = asyncio.run(orchestrator.delegate_task(
    "What accessories are in stock and which is the best value?",
    required_skill="product_research"
))
print(f"Task ID:  {task.id}")
print(f"Status:   {task.status.state.upper()}")
if task.artifacts:
    print(f"Artifact: {task.artifacts[0]['parts'][0]['text']}")
    print(f"Tools used: {task.artifacts[0].get('metadata', {}).get('tools_used', 'N/A')}")


In [ ]:
# Test 2: Complex request requiring orchestration
asyncio.run(orchestrator.handle_complex_request(
    "I want to check if the laptop is in stock, get the order status for ORD-10003, "
    "and understand the refund policy for recently ordered items."
))


💡 **Key takeaway — A2A in three steps:**

1. **Discover**: Read an Agent Card (`/.well-known/agent.json`) to learn what the agent can do
2. **Delegate**: Send a Task (`POST /tasks/send`) — the agent handles execution autonomously
3. **Receive**: Get the completed Task with Artifacts back

**MCP vs A2A, side by side:**

| | MCP | A2A |
|---|---|---|
| Connects | Agent ↔ Tools | Agent ↔ Agent |
| Direction | Vertical (down to tools) | Horizontal (across to peers) |
| Discovery | `tools/list` | Agent Card at `/.well-known/agent.json` |
| Invocation | `tools/call` | `POST /tasks/send` |
| Best for | Tool integration | Cross-framework collaboration |

They're complementary: a ResearchAgent uses **MCP** to call its search tools, while an Orchestrator uses **A2A** to delegate tasks to the ResearchAgent.


---

## 9. Architecture Comparison

Same question, two agent architectures. Watch the difference in how they arrive at the answer.


In [ ]:
q = "What's Tesla's Q4 2025 revenue, and what's that per month?"

print("▓" * 60)
print("▓  APPROACH 1: ReAct (Basic Agent Loop)")
print("▓" * 60)
t1 = time.time()
r1 = agent(q)
t1 = time.time() - t1

print("\n" + "▓" * 60)
print("▓  APPROACH 2: Plan-and-Execute")
print("▓" * 60)
t2 = time.time()
r2 = plan_and_execute(q)
t2 = time.time() - t2

print("\n" + "═" * 60)
print(f"ReAct:            {t1:.1f}s")
print(f"Plan-and-Execute: {t2:.1f}s")
print("Same answer, different approach — reactive vs deliberate.")


💡 **Key takeaway:** For simple questions (1–3 steps), ReAct is faster — no planning overhead. For complex tasks (4+ steps), Plan-and-Execute often uses fewer total tokens. Choose based on the task.


---

## 10. Practice Challenges

### Challenge 1: Multi-Source Agent
Add `get_stock_price(ticker)` and `convert_currency(amount, from_cur, to_cur)` tools. Ask: *"What's Apple's stock price in UAE dirhams?"* — the agent should chain both tools.

### Challenge 2: Reflexion for SQL
Write an evaluator that runs SQL against a test SQLite database. Use Reflexion to generate a correct query for: *"Top 3 product categories by revenue in Q3 2025."*

### Challenge 3: Extend the MCP Server
Add `create_return_request(order_id, reason)` and `get_recommendations(category, budget)` tools to `techstore_mcp_server.py`. Reconnect via `run_mcp_agent()` and test: *"Return ORD-10001 (damaged) and recommend a replacement laptop under $1500."*

### Challenge 4: A2A with a Writing Agent
Create a second specialised agent: `WriterAgent` with a `report_writing` skill. Update the orchestrator to: research via `ResearchAgent` then delegate report formatting to `WriterAgent`. Test with: *"Write a one-page product comparison report for our accessories range."*

### Challenge 5: Market MCP Server Chain
Connect `@modelcontextprotocol/server-filesystem` alongside your TechStore server. Ask the agent: *"Read the sales report, look up current inventory, and write an executive summary to summary.txt."* — it should call tools from both servers.


---

## Wrap-Up

You started with an LLM that could only follow instructions. You're ending with a complete agent engineering stack.

**What you built:**

1. **The Agent Loop** — `for` loop + `if tool_calls` = the core of every agent
2. **ReAct** — explicit Thought/Action/Observation traces for debuggability
3. **Plan-and-Execute** — separate planner from executor for multi-step tasks
4. **Reflexion** — self-critique with automated evaluation for verifiable output
5. **MCP Server** — 3 tools + 1 resource + 1 prompt, full Python implementation
6. **MCP Agent Integration** — LLM discovers and calls tools via the protocol at runtime
7. **Market MCP Server** — plugged in `server-filesystem` with zero server code
8. **A2A Protocol** — Agent Cards, Tasks, Artifacts; cross-agent delegation end-to-end

**The critical insight:** Every framework (LangGraph, CrewAI, OpenAI Agents SDK) is the same core loop with a different wrapper. MCP and A2A are the protocols that let those wrappers interoperate.

In **Section 11**, you'll use production frameworks — LangGraph for state machines, CrewAI for multi-agent teams, and OpenAI Agents SDK for handoff-based systems. Now you know exactly what's under the hood.
